# Task 1 & Task 2: Transformer Fine-Tuning and Clinical Verification
### Generative AI - Assignment #2 (Optimized for Kaggle Dual NVIDIA T4 GPUs)
This notebook trains `Qwen2.5-3B-Instruct` using 4-bit QLoRA on the prescribed **TrialGPT Criterion Annotations** dataset using the mandatory **37/8/8 patient-grouped split** (seed 42).

**Features Covered:**
- Dual GPU environment verification (2x 15GB T4 = 30GB total VRAM)
- Dataset integrity check & patient-leakage verification
- Empirical context & evidence truncation audit (Rubric B2, B4)
- Baseline (zero-shot) evaluation before fine-tuning
- Parameter-efficient 4-bit QLoRA training with gradient checkpointing
- Post-fine-tuning evaluation on held-out test split (Rubric C1, C2, C3)
- Extraction of 8 distinct failure modes (Rubric C4)
- Exporting adapters and learning curves for IEEE paper

In [ ]:
# 1. Environment & Dual T4 GPU Verification
!nvidia-smi
import torch
num_gpus = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs available: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB)")

In [ ]:
# 2. Clone repository & install dependencies
import os, sys

if not os.path.exists("genai-assignment2"):
    !git clone https://github.com/noumaan-siddiqui/genai-assignment2.git
    %cd genai-assignment2
else:
    %cd genai-assignment2
    !git pull origin main

!pip install -r requirements.txt -q
!pip install -U "bitsandbytes>=0.46.1" -q

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd()

In [ ]:
# 3. Execute and verify the 37/8/8 Patient-Grouped Split (Seed 42)
!python3 src/data/prepare_split.py

import json
with open("data/split_manifest.json") as f:
    manifest = json.load(f)

print("=== Split Manifest Verification ===")
print(f"Total patients: {manifest['patient_counts']['total_patients']}")
print(f"Train patients: {manifest['patient_counts']['train_patients']} (Annotations: {manifest['annotation_counts']['train_annotations']})")
print(f"Val patients: {manifest['patient_counts']['val_patients']} (Annotations: {manifest['annotation_counts']['val_annotations']})")
print(f"Test patients: {manifest['patient_counts']['test_patients']} (Annotations: {manifest['annotation_counts']['test_annotations']})")
print(f"Zero patient leakage verified: {manifest['leakage_checks_passed']}")

In [ ]:
# 4. Empirical Truncation & Evidence Loss Audit (Rubric B2, B4)
!python3 src/data/truncation_audit.py

with open("results/truncation_audit.json") as f:
    audit = json.load(f)

print("=== Truncation Audit Summary ===")
print(f"Max prompt words: {audit['max_total_prompt_words']} (~{audit['estimated_max_prompt_tokens']} tokens)")
print(f"Max gold sentence ID: {audit['max_sentence_id_in_dataset']}")
for t, res in audit['threshold_analysis'].items():
    print(f"  Threshold {t}: Truncated={res['truncated_records']}, Evidence Lost={res['evidence_lost_records']}")

In [ ]:
# 5. Baseline Evaluation (Before Fine-Tuning on Validation Set)
!python3 src/evaluation/evaluate_model.py \
    --base_model "Qwen/Qwen2.5-3B-Instruct" \
    --eval_data "data/val.jsonl" \
    --prefix "val_base"

In [ ]:
# 6. Supervised QLoRA Fine-Tuning (4-bit NF4 with Gradient Checkpointing)
!python3 src/models/train_lora.py \
    --model_name "Qwen/Qwen2.5-3B-Instruct" \
    --train_path "data/train.jsonl" \
    --val_path "data/val.jsonl" \
    --output_dir "checkpoints/lora_qwen3b" \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 4 \
    --learning_rate 2e-4 \
    --lora_r 16 \
    --lora_alpha 32 \
    --max_length 1024

In [ ]:
# 7. Post Fine-Tuning Evaluation on Validation Set
!python3 src/evaluation/evaluate_model.py \
    --adapter_path "checkpoints/lora_qwen3b/final_adapter" \
    --base_model "Qwen/Qwen2.5-3B-Instruct" \
    --eval_data "data/val.jsonl" \
    --prefix "val_finetuned"

In [ ]:
# 8. Final Held-Out Evaluation on Unseen Test Split (Rubric B1, B4, C1)
!python3 src/evaluation/evaluate_model.py \
    --adapter_path "checkpoints/lora_qwen3b/final_adapter" \
    --base_model "Qwen/Qwen2.5-3B-Instruct" \
    --eval_data "data/test.jsonl" \
    --prefix "test_finetuned"

In [ ]:
# 9. Compare Before vs. After Fine-Tuning Performance
import pandas as pd

with open("results/val_base_summary.json") as f:
    base_res = json.load(f)
with open("results/val_finetuned_summary.json") as f:
    ft_res = json.load(f)

comparison = {
    "Metric": ["Overall Accuracy", "Overall Macro-F1", "Inclusion Macro-F1", "Exclusion Macro-F1", "Evidence Sentence F1"],
    "Before Fine-Tuning": [
        base_res["overall_classification"]["accuracy"],
        base_res["overall_classification"]["macro_f1"],
        base_res["inclusion_classification"]["macro_f1"],
        base_res["exclusion_classification"]["macro_f1"],
        base_res["evidence_sentence_metrics"]["mean_f1"]
    ],
    "After Fine-Tuning": [
        ft_res["overall_classification"]["accuracy"],
        ft_res["overall_classification"]["macro_f1"],
        ft_res["inclusion_classification"]["macro_f1"],
        ft_res["exclusion_classification"]["macro_f1"],
        ft_res["evidence_sentence_metrics"]["mean_f1"]
    ]
}
df_comp = pd.DataFrame(comparison)
display(df_comp)

In [ ]:
# 10. Plot Learning Curves for IEEE Report
import matplotlib.pyplot as plt
os.makedirs("report/figures", exist_ok=True)

with open("logs/training_history.json") as f:
    history = json.load(f)

train_losses = [x["loss"] for x in history if "loss" in x]
eval_losses = [x["eval_loss"] for x in history if "eval_loss" in x]

plt.figure(figsize=(8, 5))
if train_losses:
    plt.plot(train_losses, label="Training Loss (QLoRA)", color="tab:blue")
if eval_losses:
    plt.plot([len(train_losses)//len(eval_losses) * (i+1) for i in range(len(eval_losses))], eval_losses, label="Validation Loss", color="tab:orange", marker="o")
plt.xlabel("Optimization Steps")
plt.ylabel("Loss")
plt.title("QLoRA Training Dynamics (Qwen2.5-3B on Dual T4)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("report/figures/loss_curve.pdf")
plt.savefig("report/figures/loss_curve.png", dpi=300)
plt.show()

In [ ]:
# 11. Display 8 Distinct Failure Modes for Qualitative Defense (Rubric C4)
with open("results/val_finetuned_failures.json") as f:
    failures = json.load(f)

print(f"=== Auditing {min(8, len(failures))} Distinct Failure Cases ===")
for i, fail in enumerate(failures[:8]):
    print(f"\n--- [Failure Case {i+1}] ({fail['failure_type']}) ---")
    print(f"Patient ID: {fail['patient_id']} | Criterion Type: {fail['criterion_type']}")
    print(f"Criterion: {fail['criterion_text']}")
    print(f"Gold: {fail['gold_eligibility']} (Sentences: {fail['gold_sentences']})")
    print(f"Predicted: {fail['pred_eligibility']} (Sentences: {fail['pred_sentences']})")
    print(f"Model Reasoning: {fail['reasoning']}")